<a href="https://colab.research.google.com/github/thorayya/online_exam_cheating_detection/blob/main/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mediapipe
!pip install deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.2 MB/s eta 0:00:00


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
from deepface import DeepFace
from google.colab import files
from google.colab.patches import cv2_imshow

In [ ]:
!wget -q -O detector.tflite -q https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite

In [ ]:
uploaded = files.upload()

Saving image.jpg to image (4).jpg
Saving video.mp4 to video.mp4


In [ ]:


BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = vision.FaceDetectorOptions(
    base_options=BaseOptions(model_asset_path='detector.tflite'),
    running_mode=VisionRunningMode.VIDEO
)

detector = vision.FaceDetector.create_from_options(options)



webcam = cv2.VideoCapture(0)

image_user = cv2.imread('image.jpg')


cheating_events = []



while webcam.isOpened():


  cheating_detected = False

  ret, frame = webcam.read()
  if not ret:
    break


  mp_image = mp.Image(
  image_format=mp.ImageFormat.SRGB,
  data=frame)

  timestamp_ms = int(time.time() * 1000)
  detection_result = detector.detect_for_video(mp_image, timestamp_ms)


  face_count = len(result.detections)



  if face_count == 0:
    cheating_detected = True
          #"reason: there is no face"
  elif face_count > 1:
    cheating_detected = True
          #"reason: there are more than one face"


  else:
    try:

      result = DeepFace.verify(img1_path=frame,img2_path=image_user)

      if not result["verified"]:
        cheating_detected = True

        #"reason: face is not the same"

    except Exception as e:
      cheating_detected = True
          #reason = f"Verification error: {str(e)}"


    if cheating_detected:
      event={
          'time': time.strftime('%Y-%m-%d %H:%M:%S'),
          'reason':''
      }

      cheating_events.append(event)

    cv2_imshow(frame)
    cv2.waitKey(1)


webcam.release()
cv2.destroyAllWindows()